In [111]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [120]:
import torch
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import os

data_dir = '/content/gdrive/MyDrive/Colab Notebooks/ML-NC-Project/data/'
batch_size = 10

train_transform = transforms.Compose([

    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

test_transform = transforms.Compose([
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

train_dataset = datasets.ImageFolder(os.path.join(data_dir, 'train'), transform=train_transform)
test_dataset = datasets.ImageFolder(os.path.join(data_dir, 'test'), transform=test_transform)

class_names = train_dataset.classes
print(class_names)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print("data loading setup complete, training and testing data loaders created")

['paper', 'rock', 'scissors']
data loading setup complete, training and testing data loaders created


In [121]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

class GestureCNN(nn.Module):
    def __init__(self):
        super(GestureCNN, self).__init__()
        # 1st convolutional block
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1) # 48x48 -> 48x48
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2) # 48x48 -> 24x24
        self.dropout1 = nn.Dropout(0.25)

        # 2nd convolutional block
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1) # 24x24 -> 24x24
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2) # 24x24 -> 12x12
        self.dropout2 = nn.Dropout(0.25)

        # 3rd convolutional block
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1) # 12x12 -> 12x12
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2) # 12x12 -> 6x6
        self.dropout3 = nn.Dropout(0.25)

        self.fc1 = nn.Linear(128 * 6 * 6, 512)
        self.bn_fc = nn.BatchNorm1d(512)
        self.fc2 = nn.Linear(512, 3)
        self.dropout_fc = nn.Dropout(0.5)

    def forward(self, x):
        x = self.dropout1(self.pool1(F.relu(self.bn1(self.conv1(x)))))
        x = self.dropout2(self.pool2(F.relu(self.bn2(self.conv2(x)))))
        x = self.dropout3(self.pool3(F.relu(self.bn3(self.conv3(x)))))

        x = x.view(-1, 128 * 6 * 6)

        x = self.dropout_fc(F.relu(self.bn_fc(self.fc1(x))))
        x = self.fc2(x)
        return x

model = GestureCNN()

model = model.to(device)

print("CNN Model Architecture:")
print(model)


device: cuda
CNN Model Architecture:
GestureCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout1): Dropout(p=0.25, inplace=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout2): Dropout(p=0.25, inplace=False)
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout3): Dropout(p=0.25, inplace=False)
  (fc1): Linear(in_features=4608, out_features=512, bias=Tr

In [122]:
import torch.optim as optim

# loss function
criterion = nn.CrossEntropyLoss()

# hyperparameters
learning_rate = 0.001
weight_decay = 1e-4
num_epochs = 10

optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

best_accuracy = 0.0
model_save_path = '/content/gdrive/MyDrive/Colab Notebooks/ML-NC-Project/best_gesture_model.pth' # Already in kernel state

print("training parameters set up: loss function, optimizer, and checkpointing variables initialized")

training parameters set up: loss function, optimizer, and checkpointing variables initialized


In [123]:
print("starting model training...")

history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

for epoch in range(1, num_epochs + 1):
    model.train()
    running_loss = 0.0
    correct_predictions_train = 0
    total_predictions_train = 0

    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # calculate training accuracy
        _, predicted = torch.max(outputs.data, 1)
        total_predictions_train += labels.size(0)
        correct_predictions_train += (predicted == labels).sum().item()

    # evaluate the model on the test set after each epoch
    model.eval()
    correct_predictions_test = 0
    total_predictions_test = 0
    test_loss = 0.0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            total_predictions_test += labels.size(0)
            correct_predictions_test += (predicted == labels).sum().item()

    # calculate average losses and accuracies
    epoch_loss = running_loss / len(train_loader)
    epoch_accuracy_train = (correct_predictions_train / total_predictions_train) * 100
    avg_test_loss = test_loss / len(test_loader)
    test_accuracy = (correct_predictions_test / total_predictions_test) * 100

    history['train_loss'].append(epoch_loss)
    history['train_acc'].append(epoch_accuracy_train)
    history['test_loss'].append(avg_test_loss)
    history['test_acc'].append(test_accuracy)

    print(f'Epoch [{epoch}/{num_epochs}], ' \
          f'Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_accuracy_train:.2f}%, ' \
          f'Test Loss: {avg_test_loss:.4f}, Test Acc: {test_accuracy:.2f}%' \
          '\n')

    # save the best model
    if test_accuracy > best_accuracy:
        best_accuracy = test_accuracy
        torch.save(model.state_dict(), model_save_path)
        print(f'saved best model with test accuracy: {best_accuracy:.2f}% to {model_save_path}')

print("training complete")

starting model training...
Epoch [1/10], Train Loss: 0.3258, Train Acc: 87.36%, Test Loss: 0.2048, Test Acc: 94.05%

saved best model with test accuracy: 94.05% to /content/gdrive/MyDrive/Colab Notebooks/ML-NC-Project/best_gesture_model.pth
Epoch [2/10], Train Loss: 0.1268, Train Acc: 96.05%, Test Loss: 0.0552, Test Acc: 97.86%

saved best model with test accuracy: 97.86% to /content/gdrive/MyDrive/Colab Notebooks/ML-NC-Project/best_gesture_model.pth
Epoch [3/10], Train Loss: 0.1086, Train Acc: 96.78%, Test Loss: 0.0218, Test Acc: 99.76%

saved best model with test accuracy: 99.76% to /content/gdrive/MyDrive/Colab Notebooks/ML-NC-Project/best_gesture_model.pth
Epoch [4/10], Train Loss: 0.0845, Train Acc: 97.33%, Test Loss: 0.0385, Test Acc: 99.05%

Epoch [5/10], Train Loss: 0.0759, Train Acc: 97.33%, Test Loss: 0.0255, Test Acc: 99.05%

Epoch [6/10], Train Loss: 0.0931, Train Acc: 97.39%, Test Loss: 0.0412, Test Acc: 98.81%

Epoch [7/10], Train Loss: 0.0571, Train Acc: 98.24%, Test Los